# 01 - 数据探索

本 Notebook 用于：
1. 加载基金净值数据（efinance）
2. 加载指数行情数据（baostock）
3. 数据质量检查与可视化

In [ ]:
import sys
sys.path.insert(0, '..')

import config
from barra import DataLoader

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 加载基金净值

In [ ]:
loader = DataLoader(config.DATA_DIR, config.START_DATE, config.END_DATE)
fund_nav = loader.load_fund_nav(config.FUND_CODE)

print(f"基金: {config.FUND_NAME} ({config.FUND_CODE})")
print(f"数据范围: {fund_nav.index[0].date()} ~ {fund_nav.index[-1].date()}")
print(f"交易日数: {len(fund_nav)}")
fund_nav.head()

In [ ]:
# 净值走势
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(fund_nav.index, fund_nav['nav'], label='累计净值')
axes[0].set_title(f'{config.FUND_NAME} 累计净值')
axes[0].legend()

axes[1].plot(fund_nav.index, fund_nav['return'] * 100, alpha=0.6)
axes[1].set_title('日收益率 (%)')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

## 2. 加载指数数据

In [ ]:
index_df = loader.load_all_indexes(config.FACTOR_INDEXES)

print("指数数据:")
print(f"  交易日数: {len(index_df)}")
print(f"  列: {list(index_df.columns)}")
index_df.head()

In [ ]:
# 指数累计收益对比
cum_returns = (1 + index_df).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
for col in cum_returns.columns:
    ax.plot(cum_returns.index, cum_returns[col], label=col)
ax.set_title('因子指数累计收益对比')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 3. 数据质量检查

In [ ]:
# 检查缺失值
print("基金净值缺失值:", fund_nav.isnull().sum().sum())
print("指数数据缺失值:")
print(index_df.isnull().sum())

# 描述统计
print("\n基金日收益率统计:")
print(fund_nav['return'].describe())